In [17]:
import pandas as pd
import numpy as np
import os
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load dataset
df = pd.read_csv('../dataset/ev_charging_patterns.csv')
df = df.drop_duplicates()
df = df.dropna()

# 2. Feature Engineering
df['SoC_Difference'] = df['State of Charge (End %)'] - df['State of Charge (Start %)']

# 3. Define Features (X) and Target (y)
X = df.drop(columns=['Energy Consumed (kWh)'])
X = X.select_dtypes(include=['int64', 'float64']) # Keep only numeric features
y = df['Energy Consumed (kWh)']

# 4. Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Data prepared successfully!")
print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

Data prepared successfully!
X_train shape: (904, 10), X_test shape: (227, 10)


In [18]:
# Initialize models
lr_model = LinearRegression()
dt_model = DecisionTreeRegressor(random_state=42)
rf_model = RandomForestRegressor(n_estimators=100, random_state=42)

# Train models
print("Training models...")
lr_model.fit(X_train, y_train)
dt_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)
print("All models trained successfully!")

Training models...
All models trained successfully!


In [19]:
# Model Evaluation
models = {
    "Linear Regression": lr_model,
    "Decision Tree": dt_model,
    "Random Forest": rf_model
}

print("================ MODEL EVALUATION ================\n")
for name, model in models.items():
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    print(f"=== {name} Evaluation ===")
    print(f"Test MAE:      {mae:.4f}")
    print(f"Test RMSE:     {rmse:.4f}")
    print(f"Test R² Score: {r2:.4f}\n")

================ MODEL EVALUATION ================

=== Linear Regression Evaluation ===
Test MAE:      19.2238
Test RMSE:     23.1609
Test R² Score: -0.0251

=== Decision Tree Evaluation ===
Test MAE:      24.2522
Test RMSE:     30.1966
Test R² Score: -0.7425

=== Random Forest Evaluation ===
Test MAE:      19.1553
Test RMSE:     23.1430
Test R² Score: -0.0235



In [21]:
# Ensure target directory exists and save model & columns
os.makedirs('../src', exist_ok=True)

joblib.dump(rf_model, '../src/ev_model.pkl')
joblib.dump(X.columns.tolist(), '../src/model_columns.pkl')

print("Best model (Random Forest) successfully saved to ../src/ev_model.pkl!")

Best model (Random Forest) successfully saved to ../src/ev_model.pkl!
